# 気象庁 過去の気象データ 一括ダウンロード

[観測所ファインダー](https://awg-yk.github.io/weather-station-finder/) で選んだ
**地点リスト**を使って、気象庁の過去の気象データをまとめて取得するノートブックです。

## 使い方
1. ページ上部の **「すべてのセルを実行」** を押します
   - 「**このノートブックは Google が作成したものではありません**」という警告が出たら、**「このまま実行」** をクリックしてください（GitHub上の安全なコードです）
2. 少し待つと、下に**入力フォーム**が表示されます（コードは非表示です）。**1 → 2 → 3** の順に選びます:
   - **1. 地点リスト**: ファインダーの「1. Colab用にコピー」を押し、貼り付け欄に貼る
   - **2. データの種類（プルダウン）＆ 観測項目（チェックボックス・複数選択可）** を選ぶ
   - **3. 期間の種類 ＆ 開始〜終了** を選ぶ
3. 「ダウンロード開始」を押すと進捗が表示され、最後にZIPが手元に落ちてきます

- 既定の期間は「1976年1月1日 〜 昨日」（細かく設定しない場合の目安）
- **気象庁の1回あたりデータ量上限を超えないよう、期間を自動で分割**して取得します
- まとめ方は「地点ごと／期間ごと／自動」。**期間ごとは複数地点を1リクエストにまとめて取得するため速い**
- **実行するたびに出力はまっさらにリセット**され、ZIPにはその回の結果だけが入ります
- 気象庁サーバーに負荷をかけないよう、1件ずつ間隔（3秒）を空けて取得します

### 10分値について
「データの種類」で **10分値** を選ぶと、時別値〜3か月別値とは別の経路でデータを取得します。
気象庁「過去の気象データ・ダウンロード」（本ノートブックが通常使うAPI）には10分値の集計期間が
無いため、姉妹サイトの **「過去の気象データ検索」(etrn)** のページを **1地点×1日ごと** に取得します。
そのため:
- 観測項目は選択できず、そのページに載っている項目をすべてそのまま取得します
- まとめ方は常に「地点ごとに1ファイル」になります
- リクエスト数が「地点数 × 日数」になるため、長い期間や多数の地点を選ぶと非常に時間がかかります。
  短い期間・少数の地点で使うことをおすすめします


In [ ]:
#@title 気象庁データ 一括ダウンロード（実行すると下にフォームが表示されます） { display-mode: "form" }
# ============================================================
# 気象庁 過去の気象データ 一括ダウンロード（Google Colab・フォーム版）
#
# 観測所ファインダー( https://awg-yk.github.io/weather-station-finder/ )で
# 出力したCSVを入力に、指定した種類・項目・期間のデータをまとめて取得します。
#
# 使い方:
#   1. ページ上部の「すべてのセルを実行」を押すとフォームが表示されます
#      （「このノートブックはGoogleが作成したものではありません」と出たら「このまま実行」をクリック）
#   2. 地点リストを渡す（どちらか）:
#        (A) ファインダーの「Colab用にコピー」を押し、フォームの「または貼付」欄に貼り付け
#        (B) 「選択結果をCSVでダウンロード」で保存したCSVを「CSVを選択」でアップロード
#   3. データの種類・観測項目・期間などをプルダウンで選択
#   4. 「ダウンロード開始」ボタンを押すと取得が進み、最後にZIPが手元に落ちてきます
#
# 特徴:
#   - すべてプルダウン等のフォームで選択（横スクロールで隠れる問題を解消）
#   - 「連続した期間」と「特定の期間を複数年分」の2モードに対応
#   - 取得後、地点ごとに1ファイルへ自動結合してファイル数を削減
#   - 地点番号(prec_no/block_no)はCSVから直接取得。取得済みファイルはスキップ再開
#   - 気象庁サーバーに負荷をかけないよう、1件ずつスリープを挟んで取得します
# ============================================================

!pip install -q requests ipywidgets

import calendar
import csv as csv_module
import io
import json
import re
import shutil
import time
from dataclasses import asdict, dataclass, field
from datetime import date, timedelta
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from urllib.parse import parse_qs, urlparse

import requests
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

ROOT_URL = "https://www.data.jma.go.jp/risk/obsdl/index.php"
SHOW_URL = "https://www.data.jma.go.jp/risk/obsdl/show/table"
STATIONS_JSON_URL = "https://raw.githubusercontent.com/awg-yk/weather-station-finder/main/data/stations.json"

# 10分値は「過去の気象データ・ダウンロード」(obsdl)のAPIには存在せず、姉妹サイトの
# 「過去の気象データ検索」(etrn)のページを1地点×1日ずつ取得してテーブルをスクレイピングする
# しかない（気象庁公式のCSV一括取得APIが対応していないため）。
ETRN_VIEW_URL = "https://www.data.jma.go.jp/stats/etrn/view/{name}.php"
TEN_MIN_VIEW_BY_PREFIX = {"s": "10min_s1", "a": "10min_a1"}  # 気象官署/アメダスでファイル名が異なる
TEN_MIN_TABLE_ID = "tablefix1"

MAX_RETRIES = 3

# 集計期間: (コード, 表示名, 連続モードでの分割単位)
PERIOD_OPTIONS: List[Tuple[str, str, str]] = [
    ("10min", "10分値",   "day"),
    ("9", "時別値",     "month"),
    ("1", "日別値",     "year"),
    ("2", "半旬別値",   "year"),
    ("4", "旬別値",     "year"),
    ("5", "月別値",     "year"),
    ("6", "3か月別値",  "year"),
]

# 時別値(9)用の観測項目: (コード, 表示名, カテゴリ)
ELEMENTS_HOURLY: List[Tuple[str, str, str]] = [
    ("201", "気温", "気温"),
    ("101", "降水量（前1時間）", "降水量"),
    ("301", "風向・風速", "風"),
    ("401", "日照時間（前1時間）", "日照時間"),
    ("610", "全天日射量（前1時間）", "日照時間"),
    ("501", "積雪の深さ", "積雪"),
    ("503", "降雪の深さ（前1時間）", "積雪"),
    ("605", "相対湿度", "湿度"),
    ("604", "蒸気圧", "湿度"),
    ("612", "露点温度", "湿度"),
    ("601", "現地気圧", "湿度"),
    ("602", "海面気圧", "湿度"),
    ("607", "雲量", ""),
    ("703", "天気", ""),
    ("704", "視程", ""),
]

# 日別/半旬/旬/月/3か月 共通の観測項目: (コード, 表示名, 対応期間集合, カテゴリ)
ELEMENTS_OTHER: List[Tuple[str, str, set, str]] = [
    ("201", "平均気温",            {"1", "2", "4", "5", "6"}, "気温"),
    ("202", "最高気温",            {"1", "2", "4", "5", "6"}, "気温"),
    ("203", "最低気温",            {"1", "2", "4", "5", "6"}, "気温"),
    ("204", "日最高気温の平均",    {"2", "4", "5", "6"}, "気温"),
    ("206", "日最低気温の平均",    {"2", "4", "5", "6"}, "気温"),
    ("205", "日最高気温の最低",    {"2", "4", "5", "6"}, "気温"),
    ("207", "日最低気温の最高",    {"2", "4", "5", "6"}, "気温"),
    ("101", "降水量の合計",        {"1", "2", "4", "5", "6"}, "降水量"),
    ("102", "日降水量の最大",      {"2", "4", "5", "6"}, "降水量"),
    ("401", "日照時間",            {"1", "2", "4", "5", "6"}, "日照時間"),
    ("402", "日照率",              {"2", "4", "5", "6"}, "日照時間"),
    ("610", "合計全天日射量",      {"1", "2", "4", "5", "6"}, "日照時間"),
    ("501", "最深積雪",            {"1", "2", "4", "5", "6"}, "積雪"),
    ("503", "降雪量の合計",        {"1", "2", "4", "5", "6"}, "積雪"),
    ("504", "降雪量日合計の最大",  {"2", "4", "5", "6"}, "積雪"),
    ("301", "平均風速",            {"1", "2", "4", "5", "6"}, "風"),
    ("302", "最大風速（風向）",    {"1", "2", "4", "5", "6"}, "風"),
    ("304", "最大瞬間風速（風向）", {"1", "2", "4", "5", "6"}, "風"),
    ("305", "最多風向",            {"1", "2", "4", "5", "6"}, "風"),
    ("605", "平均相対湿度",        {"1", "2", "4", "5", "6"}, "湿度"),
    ("606", "最小相対湿度",        {"1", "2", "4", "5", "6"}, "湿度"),
    ("604", "平均蒸気圧",          {"1", "2", "4", "5", "6"}, "湿度"),
    ("601", "平均現地気圧",        {"1", "2", "4", "5", "6"}, "湿度"),
    ("602", "平均海面気圧",        {"1", "2", "4", "5", "6"}, "湿度"),
    ("603", "最低海面気圧",        {"1", "2", "4", "5", "6"}, "湿度"),
    ("607", "平均雲量",            {"1", "2", "4", "5", "6"}, ""),
    ("620", "雪日数",              {"2", "4", "5", "6"}, ""),
    ("621", "雷日数",              {"2", "4", "5", "6"}, ""),
    ("622", "霧日数",              {"2", "4", "5", "6"}, ""),
    ("701", "天気概況（昼）",      {"1"}, ""),
    ("702", "天気概況（夜）",      {"1"}, ""),
]

ID_COLUMN_CANDIDATES = ["観測所ID", "地点コード", "地点番号"]


@dataclass
class Station:
    station_id: str
    name: str
    prec_no: str
    block_no: str
    station_type: str
    elements: set
    status: str

    def station_num(self) -> str:
        # obsdlの地点番号は block_no の桁数で接頭辞が決まる:
        #   5桁(WMO 47xxx = 気象官署) → "s"+block_no、4桁以下(アメダス) → "a"+ゼロ埋め4桁。
        # 「種別」ラベルでは判定しない（例: つくば/館野 47646 は種別アメダスでも obsdl では s47646）。
        if len(self.block_no) >= 5:
            return "s" + self.block_no
        return "a" + self.block_no.zfill(4)


@dataclass
class WeatherDataPayload:
    stationNumList: List[str] = field(default_factory=list)
    aggrgPeriod: int = 1
    elementNumList: List[List[str]] = field(default_factory=list)
    interAnnualType: int = 1
    ymdList: List[str] = field(default_factory=list)  # [y1, y2, m1, m2, d1, d2]
    optionNumList: List[Any] = field(default_factory=list)
    downloadFlag: str = "true"
    rmkFlag: int = 1
    disconnectFlag: int = 1
    youbiFlag: int = 0
    fukenFlag: int = 0
    kijiFlag: int = 0
    huukouFlag: int = 0
    csvFlag: int = 1
    jikantaiFlag: int = 0
    jikantaiList: List[Any] = field(default_factory=list)
    ymdLiteral: int = 1

    def to_post_data(self) -> dict:
        data = {}
        for key, value in asdict(self).items():
            data[key] = json.dumps(value) if isinstance(value, list) else value
        return data


# ------------------------------------------------------------
# 入力CSVの読み込み
# ------------------------------------------------------------
def parse_prec_block_from_url(url: str) -> Tuple[Optional[str], Optional[str]]:
    if not url:
        return None, None
    try:
        q = parse_qs(urlparse(url).query)
        return q.get("prec_no", [None])[0], q.get("block_no", [None])[0]
    except Exception:
        return None, None


def parse_elements_cell(cell: str) -> set:
    if not cell:
        return set()
    return {p.strip() for p in re.split(r"[\/／,、]", cell) if p.strip()}


def load_master_fallback() -> Dict[str, dict]:
    cache = Path("stations_master.json")
    if not cache.exists():
        resp = requests.get(STATIONS_JSON_URL, timeout=30)
        resp.raise_for_status()
        cache.write_bytes(resp.content)
    raw = json.loads(cache.read_text(encoding="utf-8"))
    return {str(s["id"]): s for s in raw["stations"]}


def find_id_column(fieldnames: List[str]) -> Optional[str]:
    for cand in ID_COLUMN_CANDIDATES:
        if cand in fieldnames:
            return cand
    return None


def read_stations_from_text(text: str) -> List[Station]:
    text = text.lstrip("﻿")  # 貼り付け時に残るBOMを除去（列名の頭に付くと列検出に失敗するため）
    reader = csv_module.DictReader(io.StringIO(text))
    fields = reader.fieldnames or []
    id_col = find_id_column(fields)
    if id_col is None:
        raise ValueError(f"CSVに地点IDの列（{'／'.join(ID_COLUMN_CANDIDATES)}）が見つかりません。列: {fields}")
    rows = list(reader)

    has_url = "気象庁ページURL" in fields
    master = None if has_url else load_master_fallback()

    out: List[Station] = []
    for row in rows:
        sid = (row.get(id_col) or "").strip()
        if not sid:
            continue
        name = (row.get("地点名") or "").strip()
        stype = (row.get("種別") or "").strip()
        status = (row.get("状態") or "").strip()
        elems = parse_elements_cell(row.get("観測要素", ""))

        prec = block = None
        if has_url:
            prec, block = parse_prec_block_from_url(row.get("気象庁ページURL", ""))
        if (not prec or not block) and master is not None:
            m = master.get(sid)
            if m:
                prec, block = m.get("precNo"), m.get("blockNo")
                if not stype:
                    stype = m.get("stationType", "")
        if not prec or not block:
            print(f"  [スキップ] {name or sid}: 地点番号を特定できません")
            continue
        if not stype:
            stype = "気象官署" if len(block) >= 5 and block.startswith("47") else "アメダス"
        out.append(Station(sid, name or sid, prec, block, stype, elems, status))
    return out


# ------------------------------------------------------------
# 期間の分割（連続モードのみ。複数年モードは分割せず1リクエスト）
# ------------------------------------------------------------
def date_chunks(start: date, end: date, unit: str):
    cur = start
    while cur <= end:
        if unit == "month":
            if cur.month == 12:
                last = date(cur.year, 12, 31)
            else:
                last = date(cur.year, cur.month + 1, 1) - timedelta(days=1)
            chunk_end = min(last, end)
            nxt_month = cur.month + 1
            nxt_year = cur.year + (1 if nxt_month > 12 else 0)
            nxt_month = 1 if nxt_month > 12 else nxt_month
            nxt = date(nxt_year, nxt_month, 1)
        else:
            chunk_end = min(date(cur.year, 12, 31), end)
            nxt = date(cur.year + 1, 1, 1)
        yield cur, chunk_end
        cur = nxt


# ------------------------------------------------------------
# 1リクエストのデータ量が気象庁の上限を超えないよう分割する
#   気象庁の判定式（top.2.1.js より）:
#     地点数 × 項目数 × 期間の点数(nOfPr) × 重み ≤ seigen(=44000)
#   本ツールは1リクエスト=1地点なので、項目数 × 点数 × 重み ≤ 上限 になるよう分割する。
# ------------------------------------------------------------
SEIGEN = 44000
VOLUME_LIMIT = 40000  # 44000に対して余裕を持たせた実効上限


def _safe_date(y: int, m: int, d: int) -> date:
    last = calendar.monthrange(y, m)[1]
    return date(y, m, min(d, last))


def _days_between(y1, m1, d1, y2, m2, d2) -> int:
    return abs((_safe_date(y2, m2, d2) - _safe_date(y1, m1, d1)).days) + 1


def count_periods(aggrg_type: int, inter: int, y1, y2, m1, m2, d1, d2) -> int:
    """気象庁 countPrNum を移植。期間の点数(nOfPr)を返す。ymd=[y1,y2,m1,m2,d1,d2]。"""
    if inter == 1:  # 連続した期間
        if aggrg_type in (1, 8, 9):
            diff = _days_between(y1, m1, d1, y2, m2, d2)
            if aggrg_type == 9:
                diff *= 24
        elif aggrg_type in (2, 4):
            sub = 6 if aggrg_type == 2 else 3
            diff = abs((y2 - y1) * 12 * sub + (m2 - m1) * sub + (d2 - d1)) + 1
        elif aggrg_type in (5, 6):
            diff = abs(y2 * 12 + m2 - y1 * 12 - m1) + 1
        else:
            diff = _days_between(y1, m1, d1, y2, m2, d2)
    else:  # 特定の期間を複数年分
        if aggrg_type in (1, 8, 9):
            dt1 = _safe_date(y1, m2, d2)
            dt2 = _safe_date(y1, m1, d1)
            if dt1 < dt2:
                dt1 = _safe_date(y1 + 1, m2, d2)
            diff_day = abs((dt1 - dt2).days) + 1
            diff_year = abs(y2 - y1) + 1
            diff = diff_year * diff_day
            if aggrg_type == 9:
                diff *= 24
        elif aggrg_type in (2, 4):
            sub = 6 if aggrg_type == 2 else 3
            yd = abs(y1 - y2) + 1
            if m1 < m2 or (m1 == m2 and d1 <= d2):
                md = abs((m2 - m1) * sub + (d2 - d1)) + 1
            else:
                md = abs(12 * sub - ((m1 - m2) * sub + (d1 - d2))) + 1
            diff = yd * md
        elif aggrg_type in (5, 6):
            yd = abs(y1 - y2) + 1
            md = (m2 - m1 + 1) if m1 <= m2 else (12 - (m1 - m2) + 1)
            diff = yd * md
        else:
            diff = abs(y2 - y1) + 1
    return int(diff)


# 半旬別値(2)/旬別値(4)/月別値(5)/3か月別値(6)は「日」が実際の暦日ではなく
# 月内の区分index（半旬=1-6、旬=1-3、月・3か月=常に1固定）なので、date()で年またぎの
# 連続期間を分割すると（例: 12月31日をそのまま使ってしまう）範囲外の値を気象庁へ送って
# しまう。実際の気象庁側JS(top.2.1.js)でも月別/3か月別はinid/enddを常に1に固定し、
# 半旬/旬は1-6・1-3にクランプしている（calcDayValue）。ここでは暦日を経由せず、
# 「月内の区分index」を1本の整数（線形index）に変換して年境界の分割・二分探索を行う。
def _period_sub(aggrg_type: int) -> int:
    if aggrg_type == 2:
        return 6
    if aggrg_type == 4:
        return 3
    return 1  # 5, 6（月別値・3か月別値は常にd=1固定）


def _to_linear(y: int, m: int, d: int, sub: int) -> int:
    return ((y * 12) + (m - 1)) * sub + (d - 1)


def _from_linear(idx: int, sub: int) -> Tuple[int, int, int]:
    d = idx % sub + 1
    total_months = idx // sub
    y = total_months // 12
    m = total_months % 12 + 1
    return y, m, d


def _index_year_chunks(start_idx: int, end_idx: int, sub: int):
    """半旬/旬/月/3か月別値の連続期間を、年境界（12月の最終区分）で分割する。"""
    cur = start_idx
    while cur <= end_idx:
        cy, _cm, _cd = _from_linear(cur, sub)
        year_end_idx = _to_linear(cy, 12, sub, sub)
        chunk_end = min(year_end_idx, end_idx)
        yield cur, chunk_end
        cur = chunk_end + 1


def build_plan(inter_type, aggrg_type, n_el, y1, m1, d1, y2, m2, d2, chunk_unit):
    """各リクエストが上限内に収まる計画を作る。戻り値: [(inter, [Y1,Y2,M1,M2,D1,D2], 名前suffix), ...]"""
    weight = 1.5 if aggrg_type == 8 else 1.0
    use_index = aggrg_type in (2, 4, 5, 6)  # 半旬/旬/月/3か月別値は「日」が暦日ではないため
    sub = _period_sub(aggrg_type) if use_index else None

    def cost(inter, Y1, Y2, M1, M2, D1, D2):
        return n_el * weight * count_periods(aggrg_type, inter, Y1, Y2, M1, M2, D1, D2)

    def split_cont_date(cs, ce):
        # 連続期間（暦日）を、上限を超えるなら日数で二分して収める
        if cost(1, cs.year, ce.year, cs.month, ce.month, cs.day, ce.day) <= VOLUME_LIMIT or cs >= ce:
            return [("1", [cs.year, ce.year, cs.month, ce.month, cs.day, ce.day], f"{cs.isoformat()}_{ce.isoformat()}")]
        mid = cs + (ce - cs) // 2
        return split_cont_date(cs, mid) + split_cont_date(mid + timedelta(days=1), ce)

    def split_cont_idx(cs_idx, ce_idx):
        # 連続期間（半旬/旬/月/3か月の区分index）を、上限を超えるなら区分数で二分して収める
        cy1, cm1, cd1 = _from_linear(cs_idx, sub)
        cy2, cm2, cd2 = _from_linear(ce_idx, sub)
        if cost(1, cy1, cy2, cm1, cm2, cd1, cd2) <= VOLUME_LIMIT or cs_idx >= ce_idx:
            name = f"{cy1}{cm1:02d}{cd1:02d}_{cy2}{cm2:02d}{cd2:02d}"
            return [("1", [cy1, cy2, cm1, cm2, cd1, cd2], name)]
        mid = cs_idx + (ce_idx - cs_idx) // 2
        return split_cont_idx(cs_idx, mid) + split_cont_idx(mid + 1, ce_idx)

    specs = []
    if inter_type == "1":
        if use_index:
            start_idx = _to_linear(y1, m1, d1, sub)
            end_idx = _to_linear(y2, m2, d2, sub)
            for cs, ce in _index_year_chunks(start_idx, end_idx, sub):
                specs.extend(split_cont_idx(cs, ce))
        else:
            d1c = min(d1, calendar.monthrange(y1, m1)[1])
            d2c = min(d2, calendar.monthrange(y2, m2)[1])
            for cs, ce in date_chunks(date(y1, m1, d1c), date(y2, m2, d2c), chunk_unit):
                specs.extend(split_cont_date(cs, ce))
    else:
        d1c = min(d1, calendar.monthrange(2000, m1)[1])
        d2c = min(d2, calendar.monthrange(2000, m2)[1])
        per_year = cost(2, 2000, 2000, m1, m2, d1c, d2c)  # 1年分の量
        # 年をまたぐ期間（例: 各年10/1〜5/31）が複数年分1ファイルに混ざらないよう、
        # 「まとめ方」に関わらず必ず1年（サイクル）ごとに1リクエスト＝1ファイルにする。
        if per_year <= VOLUME_LIMIT:
            # 年をまたぐ期間（m1 > m2）は各サイクルが Y年〜(Y+1)年のデータを使うため、
            # 終了年 y2 を新たなサイクルの開始年にはしない（y2+1年のデータは存在しないため）。
            # y1==y2（単年選択）でも最低1サイクルは出力する。
            last_start_year = max(y1, y2 - 1) if m1 > m2 else y2
            for Y in range(y1, last_start_year + 1):
                suffix = f"{Y}{'-' + str(Y + 1) if m1 > m2 else ''}_各年{m1:02d}{d1c:02d}-{m2:02d}{d2c:02d}"
                specs.append(("2", [Y, Y, m1, m2, d1c, d2c], suffix))
        else:
            # 1年分でも上限超 → 各年を連続期間として分割
            for Y in range(y1, y2 + 1):
                if use_index:
                    ys_idx = _to_linear(Y, m1, d1c, sub)
                    ye_idx = _to_linear(Y, m2, d2c, sub)
                    for cs, ce in _index_year_chunks(ys_idx, ye_idx, sub):
                        specs.extend(split_cont_idx(cs, ce))
                else:
                    ys = min(d1, calendar.monthrange(Y, m1)[1])
                    ye = min(d2, calendar.monthrange(Y, m2)[1])
                    for cs, ce in date_chunks(date(Y, m1, ys), date(Y, m2, ye), chunk_unit):
                        specs.extend(split_cont_date(cs, ce))
    return specs


# ------------------------------------------------------------
# 取得（リトライ＋エラーHTML検出つき）
# ------------------------------------------------------------
def looks_like_csv(content: bytes) -> bool:
    head = content[:200].lstrip()
    if head[:1] == b"<":
        return False
    return len(content) > 0


def fetch_data(session, station_nums, ymd, aggrg_period, elements, inter_annual_type, sleep_sec):
    payload = WeatherDataPayload(
        stationNumList=list(station_nums),
        aggrgPeriod=int(aggrg_period),
        elementNumList=[[code, ""] for code in elements],
        interAnnualType=int(inter_annual_type),
        ymdList=[str(x) for x in ymd],
    )
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.post(SHOW_URL, data=payload.to_post_data(),
                                headers={"Referer": ROOT_URL}, timeout=60)
            resp.raise_for_status()
            if not looks_like_csv(resp.content):
                raise RuntimeError("CSV以外の応答（混雑またはデータ量超過の可能性）")
            return resp.content
        except Exception as e:
            last_err = e
            if attempt < MAX_RETRIES:
                time.sleep(sleep_sec * attempt)
    raise last_err


def save_csv(content: bytes, output_path: Path) -> None:
    try:
        output_path.write_text(content.decode("cp932"), encoding="utf-8-sig", newline="")
    except UnicodeDecodeError:
        output_path.write_bytes(content)


# ------------------------------------------------------------
# 10分値の取得（etrnページのHTMLテーブルをスクレイピング）
#   obsdlのCSV APIには10分値の集計期間が無いため、「過去の気象データ検索」(etrn)の
#   1地点・1日ぶんの表示ページ（view/10min_s1.php・10min_a1.php）を1リクエストずつ
#   取得し、テーブルをCSVに変換する。地点ごとの観測項目（列構成）が異なるため、
#   列を固定せずヘッダー行から動的に読み取る。
# ------------------------------------------------------------
from html.parser import HTMLParser


class _TableExtractor(HTMLParser):
    """指定idの<table>だけを、colspanを展開したセル文字列の行リストに変換する簡易パーサー。"""

    def __init__(self, target_id: str):
        super().__init__()
        self.target_id = target_id
        self.in_target = False
        self.depth = 0
        self.rows: List[List[Tuple[str, int]]] = []
        self._cur_row: Optional[List[Tuple[str, int]]] = None
        self._cur_cell: Optional[List[str]] = None
        self._cur_colspan = 1

    def handle_starttag(self, tag, attrs):
        attrs_d = dict(attrs)
        if tag == "table":
            if not self.in_target and attrs_d.get("id") == self.target_id:
                self.in_target = True
                self.depth = 1
            elif self.in_target:
                self.depth += 1
        elif self.in_target and tag == "tr":
            self._cur_row = []
        elif self.in_target and tag in ("td", "th"):
            self._cur_cell = []
            try:
                self._cur_colspan = max(1, int(attrs_d.get("colspan", "1")))
            except ValueError:
                self._cur_colspan = 1
        elif self.in_target and tag == "br" and self._cur_cell is not None:
            self._cur_cell.append(" ")

    def handle_endtag(self, tag):
        if not self.in_target:
            return
        if tag == "table":
            self.depth -= 1
            if self.depth == 0:
                self.in_target = False
        elif tag == "tr" and self._cur_row is not None:
            self.rows.append(self._cur_row)
            self._cur_row = None
        elif tag in ("td", "th") and self._cur_cell is not None and self._cur_row is not None:
            text = "".join(self._cur_cell).strip()
            self._cur_row.append((text, self._cur_colspan))
            self._cur_cell = None

    def handle_data(self, data):
        if self.in_target and self._cur_cell is not None:
            self._cur_cell.append(data)


def extract_table_rows(html_text: str, table_id: str) -> List[List[str]]:
    """table id=table_id の中身を、colspanを展開したセル文字列の2次元リストにして返す。"""
    parser = _TableExtractor(table_id)
    parser.feed(html_text)
    grid = []
    for row in parser.rows:
        cells: List[str] = []
        for text, colspan in row:
            cells.append(text)
            cells.extend([""] * (colspan - 1))
        if cells:
            grid.append(cells)
    return grid


def _csv_escape(value: str) -> str:
    v = value or ""
    if any(c in v for c in (",", '"', "\n")):
        return '"' + v.replace('"', '""') + '"'
    return v


_TIME_CELL_RE = re.compile(r"^\d{1,2}:\d{2}$")


def fetch_10min_day(session, station: "Station", y: int, m: int, d: int) -> Optional[Tuple[List[str], List[List[str]]]]:
    """1地点・1日ぶんの10分値を取得する。戻り値: (列ラベル, データ行のリスト)。データが無ければNone。"""
    prefix = station.station_num()[0]
    view = TEN_MIN_VIEW_BY_PREFIX.get(prefix, "10min_a1")
    url = ETRN_VIEW_URL.format(name=view)
    params = {"prec_no": station.prec_no, "block_no": station.block_no,
              "year": y, "month": m, "day": d, "view": ""}
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.get(url, params=params, headers={"Referer": ROOT_URL}, timeout=60)
            resp.raise_for_status()
            html_text = resp.content.decode("cp932", errors="replace")
            grid = extract_table_rows(html_text, TEN_MIN_TABLE_ID)
            data_rows = [r for r in grid if r and _TIME_CELL_RE.match(r[0].strip())]
            if not data_rows:
                return None  # その日はデータなし（欠測またはページが空）
            header_rows = [r for r in grid if r and not _TIME_CELL_RE.match(r[0].strip())]
            n_cols = len(data_rows[0])
            labels = []
            for c in range(n_cols):
                parts = [hr[c].strip() for hr in header_rows if c < len(hr) and hr[c].strip()]
                labels.append("/".join(dict.fromkeys(parts)) if parts else f"col{c}")
            return labels, data_rows
        except Exception as e:
            last_err = e
            if attempt < MAX_RETRIES:
                time.sleep(SLEEP_SEC * attempt)
    raise last_err


def dates_in_range(y1: int, m1: int, d1: int, y2: int, m2: int, d2: int):
    d1c = min(d1, calendar.monthrange(y1, m1)[1])
    d2c = min(d2, calendar.monthrange(y2, m2)[1])
    cur, end = date(y1, m1, d1c), date(y2, m2, d2c)
    while cur <= end:
        yield cur
        cur += timedelta(days=1)


def dates_multi_year(y1: int, y2: int, m1: int, d1: int, m2: int, d2: int):
    """「特定の期間を複数年分」モード用。年をまたぐ期間（例: 各年10/1〜5/31）にも対応。"""
    for Y in range(y1, y2 + 1):
        d1c = min(d1, calendar.monthrange(Y, m1)[1])
        end_year = Y + (1 if m1 > m2 else 0)
        d2c = min(d2, calendar.monthrange(end_year, m2)[1])
        cur, end = date(Y, m1, d1c), date(end_year, m2, d2c)
        while cur <= end:
            yield cur
            cur += timedelta(days=1)


# ------------------------------------------------------------
# CSV結合・分割ユーティリティ
#   気象庁CSVは先頭に数行のヘッダー（時刻・地点名・項目名など）があり、
#   データ行は「年で始まる行」。1行目は「日付ラベル列 + (地点ごとに同じ幅の列ブロック)」
#   という構造になっている（地点ごとの列幅はその回答内で共通）。
#
#   取得は常に「地点をなるべくまとめてリクエスト数を最小化する」方式で行い、
#   出力（地点ごと／期間ごと／1ファイル）はここにある関数で取得後にローカル生成する。
# ------------------------------------------------------------
def _read_lines(path: Path) -> List[str]:
    return path.read_text(encoding="utf-8-sig").splitlines()


def _split_header_data(lines: List[str]) -> Tuple[List[str], List[str]]:
    di = next((k for k, ln in enumerate(lines) if ln[:1].isdigit()), len(lines))
    return lines[:di], lines[di:]


def _station_slice(lines: List[str], station_index: int, n_stations: int) -> List[str]:
    """1レスポンス内のn_stations地点ぶんの列から、station_index番目の地点の列だけを取り出す。"""
    out = []
    for ln in lines:
        parts = ln.split(",")
        n_fields = len(parts) - 1  # 先頭の日付/ラベル列を除いた列数
        block = n_fields // n_stations if n_stations > 0 else n_fields
        start = 1 + station_index * block
        end = start + block
        out.append(",".join([parts[0]] + parts[start:end]))
    return out


def merge_columns(lines_list: List[List[str]]) -> List[str]:
    """同じ期間・異なる地点グループの複数レスポンス（行は共通）を列（地点）方向に結合する。"""
    if len(lines_list) == 1:
        return lines_list[0]
    n_rows = min(len(ls) for ls in lines_list)
    merged_rows = []
    for i in range(n_rows):
        merged = lines_list[0][i]
        for ls in lines_list[1:]:
            rest = ls[i].split(",", 1)[1] if "," in ls[i] else ""
            if rest:
                merged = f"{merged},{rest}"
        merged_rows.append(merged)
    return merged_rows


def merge_rows(lines_list: List[List[str]]) -> List[str]:
    """同じ地点（列構成）・異なる時期チャンクの複数レスポンスを行（日付）方向に結合する。"""
    header_lines: Optional[List[str]] = None
    data_all: List[str] = []
    for lines in lines_list:
        h, d = _split_header_data(lines)
        if header_lines is None:
            header_lines = h
        data_all.extend(d)
    return (header_lines or []) + data_all


def write_lines(lines: List[str], out_path: Path) -> None:
    out_path.write_text("\r\n".join(lines) + "\r\n", encoding="utf-8-sig", newline="")


# ============================================================
# フォームUI（ipywidgets）
# ============================================================
CODE_TO_CATEGORY = {}
for v, lbl, cat in ELEMENTS_HOURLY:
    CODE_TO_CATEGORY[("9", v)] = cat
for v, lbl, kikan, cat in ELEMENTS_OTHER:
    for k in kikan:
        CODE_TO_CATEGORY[(k, v)] = cat

_this_year = date.today().year
_years = [str(y) for y in range(_this_year, 1899, -1)]
_months = [str(m) for m in range(1, 13)]
_days = [str(d) for d in range(1, 32)]
_pentads = [str(d) for d in range(1, 7)]   # 半旬別値: 月内の第何半旬か(1-6)
_decad_names = ["上旬", "中旬", "下旬"]     # 旬別値: 月内の第何旬か(1-3)
_decad_options = [(name, str(i + 1)) for i, name in enumerate(_decad_names)]
_yesterday = date.today() - timedelta(days=1)  # 気象庁データは前日ぶんまでが目安

# データの種類ごとに、開始/終了の「日」欄の意味が気象庁本家と同じになるよう切り替える。
#   day    = 実際の日(1-31)　… 時別値・日別値
#   pentad = 月内の第何半旬か(1-6) … 半旬別値
#   decad  = 月内の第何旬か(上旬/中旬/下旬) … 旬別値
#   none   = 日欄なし(年月のみ)    … 月別値・3か月別値
PERIOD_SUBMODE = {"10min": "day", "9": "day", "1": "day", "2": "pentad", "4": "decad", "5": "none", "6": "none"}
SUBMODE_OPTIONS = {"day": _days, "pentad": _pentads, "decad": _decad_options, "none": ["1"]}
SUBMODE_SUFFIX = {"day": "日", "pentad": "半旬", "decad": "", "none": ""}
SUBMODE_PREFIX = {"day": "", "pentad": "第", "decad": "", "none": ""}

# 3か月別値のとき、月の選択肢を気象庁本家と同じ「11～1」「12～2」…「10～12」表記にする
_month_options_plain = _months
_month_options_quarter = [(f"{m}～{((m + 1) % 12) + 1}", str(m)) for m in range(1, 13)]


def _decad_name(idx: int) -> str:
    return _decad_names[max(0, min(2, idx - 1))]


def _quarter_label(m: int) -> str:
    end = ((m + 1) % 12) + 1
    return f"{m}～{end}月"


def _day_to_pentad(d: int) -> int:
    return min(6, (d - 1) // 5 + 1)


def _day_to_decad(d: int) -> int:
    return min(3, (d - 1) // 10 + 1)


def _pentad_to_day(p: int) -> int:
    return (p - 1) * 5 + 1


def _decad_to_day(q: int) -> int:
    return (q - 1) * 10 + 1


def _convert_day_value(v: int, old_mode: str, new_mode: str) -> int:
    """開始/終了の「日」欄の値を、集計期間の切り替えに合わせて変換する。"""
    if old_mode == new_mode:
        return v
    if old_mode == "pentad":
        v = _pentad_to_day(v)
    elif old_mode == "decad":
        v = _decad_to_day(v)
    elif old_mode == "none":
        v = 1
    if new_mode == "pentad":
        return _day_to_pentad(v)
    if new_mode == "decad":
        return _day_to_decad(v)
    if new_mode == "none":
        return 1
    return min(v, 31)

BOX_WIDTH = "560px"
_row = lambda *w: widgets.HBox(list(w))

style = {"description_width": "96px"}
LW = widgets.Layout(width="150px")

paste_area = widgets.Textarea(
    placeholder="観測所ファインダーで「Colab用にコピー」を押し、ここに貼り付け（Ctrl+V）。",
    description="地点リスト", style=style,
    layout=widgets.Layout(width=BOX_WIDTH, height="100px"))
mode_dd = widgets.RadioButtons(
    options=[("連続した期間で表示する", "1"), ("特定の期間を複数年分、表示する", "2")],
    value="1", description="期間の種類", style=style, layout=widgets.Layout(width=BOX_WIDTH))
period_dd = widgets.Dropdown(
    options=[(lbl, code) for code, lbl, _u in PERIOD_OPTIONS],
    value="1", description="データの種類", style=style, layout=widgets.Layout(width="300px"))
# 観測項目はチェックボックス（複数選択可）。データの種類に応じて中身を作り直す。
elem_label = widgets.HTML("<div style='font-size:13px;margin-top:4px'>観測項目（複数選択できます）</div>")
elem_box = widgets.Box(layout=widgets.Layout(
    display="flex", flex_flow="row wrap", width=BOX_WIDTH,
    border="1px solid #ddd", padding="4px"))
elem_checks = []  # [(コード, Checkbox)]
# 10分値は「過去の気象データ検索」(etrn)のページに載っている項目をすべてそのまま取得するため、
# 項目の個別選択はできない（obsdlのAPIとは別の取得経路のため項目コードでの絞り込みが効かない）。
elem_note_10min = widgets.HTML(
    "<div style='font-size:12.5px;color:#555;padding:4px 0'>"
    "10分値は項目を選べません。気象庁「過去の気象データ検索」のページに表示されている項目"
    "（気圧・気温・降水量・風向風速・日照時間など、地点により異なります）をすべてそのまま取得します。"
    "</div>")

# 既定: 開始 1976/1/1（多くのアメダスの観測開始時期）〜 終了は昨日
syear = widgets.Dropdown(options=_years, value="1976", layout=LW)
smonth = widgets.Dropdown(options=_months, value="1", layout=LW)
sday = widgets.Dropdown(options=_days, value="1", layout=widgets.Layout(width="70px"))
eyear = widgets.Dropdown(options=_years, value=str(_yesterday.year), layout=LW)
emonth = widgets.Dropdown(options=_months, value=str(_yesterday.month), layout=LW)
eday = widgets.Dropdown(options=_days, value=str(_yesterday.day), layout=widgets.Layout(width="70px"))
# 半旬別値/旬別値のとき「第◯半旬」のように前後に単位を添える（気象庁本家の表記に合わせる）
sday_prefix = widgets.Label("", layout=widgets.Layout(width="16px"))
sday_suffix = widgets.Label("", layout=widgets.Layout(width="40px"))
eday_prefix = widgets.Label("", layout=widgets.Layout(width="16px"))
eday_suffix = widgets.Label("", layout=widgets.Layout(width="40px"))

period_desc = widgets.HTML()
start_row = _row(widgets.Label("開始", layout=widgets.Layout(width="40px")), syear,
                  widgets.Label("年"), smonth, widgets.Label("月"), sday_prefix, sday, sday_suffix)
end_row = _row(widgets.Label("終了", layout=widgets.Layout(width="40px")), eyear,
                widgets.Label("年"), emonth, widgets.Label("月"), eday_prefix, eday, eday_suffix)

SLEEP_SEC = 3.0  # 気象庁サーバーへの配慮のため固定
merge_mode = widgets.RadioButtons(
    options=[("自動（ファイルが少ない方でまとめる）", "auto"),
             ("地点ごとに1ファイル", "station"),
             ("期間ごとに1ファイル", "period"),
             ("常に1ファイルにまとめる", "single")],
    value="auto", description="まとめ方", style=style, layout=widgets.Layout(width=BOX_WIDTH))
merge_hint = widgets.HTML(
    "<span style='color:#555;font-size:12px'>※ 気象庁への問い合わせは、まとめ方の選択に関わらず常に地点をなるべく"
    "まとめてリクエスト数を最小化して取得します。ここで選ぶのは、取得後にローカルでファイルをどう分けるかです。"
    "「地点ごと」＝1地点につき全期間を結合（ファイル数＝地点数）、"
    "「期間ごと」＝1つの期間区分につき全地点を結合（ファイル数＝期間の区分数）、"
    "「常に1ファイルにまとめる」＝全地点・全期間を1つのCSVに結合します。「自動」はファイルが少ない方を選びます。<br>"
    "※ 期間の区分単位は、<b>時別値のみ1か月ごと</b>、それ以外（日別値・半旬別値・旬別値・"
    "月別値・3か月別値）は<b>1年ごと</b>です（連続した期間モードの場合。特定の期間を複数年分モードでは、"
    "年をまたぐ期間（例: 各年10/1〜5/31）も含め常に1年ごとに区分します）。</span>")
merge_note_10min = widgets.HTML(
    "<span style='color:#555;font-size:12px'>※ 10分値は「過去の気象データ・ダウンロード」ではなく"
    "気象庁「過去の気象データ検索」のページを<b>1地点×1日ごと</b>に取得するため、まとめ方に関わらず"
    "常に<b>地点ごとに1ファイル</b>にまとめます。地点数×日数ぶんのリクエストが必要になるので、"
    "期間や地点を絞って利用することをおすすめします。</span>")
run_btn = widgets.Button(description="ダウンロード開始", button_style="success",
                         layout=widgets.Layout(width="200px"))
# 進捗表示（ボタンのすぐ下に置き、下までスクロールしなくても状況が分かるようにする・③）
progress = widgets.IntProgress(value=0, min=0, max=1, description="進捗",
                               bar_style="info", style=style,
                               layout=widgets.Layout(width=BOX_WIDTH, visibility="hidden"))
status_label = widgets.HTML(value="")
out = widgets.Output(layout=widgets.Layout(border="1px solid #ccc", padding="6px",
                                            max_height="360px", overflow="auto"))


def refresh_elements(*_):
    pc = period_dd.value
    is_10min = pc == "10min"
    elem_label.layout.display = "none" if is_10min else ""
    elem_box.layout.display = "none" if is_10min else ""
    elem_note_10min.layout.display = "" if is_10min else "none"
    merge_mode.layout.display = "none" if is_10min else ""
    merge_hint.layout.display = "none" if is_10min else ""
    merge_note_10min.layout.display = "" if is_10min else "none"
    if is_10min:
        elem_checks.clear()
        elem_box.children = ()
        return
    if pc == "9":
        opts = [(lbl, v) for v, lbl, cat in ELEMENTS_HOURLY]
    else:
        opts = [(lbl, v) for v, lbl, kikan, cat in ELEMENTS_OTHER if pc in kikan]
    # 既定で気温系を1つ選択（なければ先頭）
    default = next((v for (lbl, v) in opts if "気温" in lbl), (opts[0][1] if opts else None))
    elem_checks.clear()
    boxes = []
    for lbl, v in opts:
        cb = widgets.Checkbox(value=(v == default), description=lbl, indent=False,
                              layout=widgets.Layout(width="180px", margin="0"))
        elem_checks.append((v, cb))
        boxes.append(cb)
    elem_box.children = tuple(boxes)


def refresh_desc(*_):
    if mode_dd.value == "1":
        period_desc.value = ("<span style='color:#555'>▼ <b>連続した期間</b>：開始日から終了日まで通しで取得します。</span>")
        start_row.children[0].value = "開始"
        end_row.children[0].value = "終了"
    else:
        period_desc.value = ("<span style='color:#555'>▼ <b>特定の期間を複数年分</b>：各年の「開始（月・日）〜終了（月・日）」を、"
                             "開始年〜終了年の各年について取得します（年の値が年範囲、月日が毎年の対象期間）。</span>")
        start_row.children[0].value = "開始"
        end_row.children[0].value = "終了"


_period_submode = {"value": "day"}  # 直前のモード（切り替え時の日⇔半旬/旬の変換に使う）


def refresh_period_fields(*_):
    new_mode = PERIOD_SUBMODE.get(period_dd.value, "day")
    old_mode = _period_submode["value"]
    new_opts = SUBMODE_OPTIONS[new_mode]
    for dd, prefix, suf in ((sday, sday_prefix, sday_suffix), (eday, eday_prefix, eday_suffix)):
        new_v = _convert_day_value(int(dd.value), old_mode, new_mode)
        dd.options = new_opts
        dd.value = str(new_v)
        dd.layout.display = "none" if new_mode == "none" else ""
        prefix.value = SUBMODE_PREFIX[new_mode]
        prefix.layout.display = "none" if new_mode == "none" else ""
        suf.value = SUBMODE_SUFFIX[new_mode]
        suf.layout.display = "none" if new_mode == "none" else ""
    _period_submode["value"] = new_mode

    # 3か月別値のときだけ、月の選択肢を「11～1」のような範囲表記にする
    month_opts = _month_options_quarter if period_dd.value == "6" else _month_options_plain
    for mdd in (smonth, emonth):
        old_v = mdd.value
        mdd.options = month_opts
        mdd.value = old_v


# 「選択された期間」プレビュー（気象庁本家の表記に合わせた文言）
selected_period_label = widgets.HTML()


def refresh_period_summary(*_):
    pc = period_dd.value
    period_label = dict((c, l) for c, l, _u in PERIOD_OPTIONS)[pc]
    sy, sm, sd = syear.value, int(smonth.value), sday.value
    ey, em, ed = eyear.value, int(emonth.value), eday.value

    if pc == "2":  # 半旬別値
        if mode_dd.value == "1":
            text = f"{sy}年 {sm}月 第{sd}半旬から<br>{ey}年 {em}月 第{ed}半旬まで の半旬別値を表示"
        else:
            text = f"{sm}月 第{sd}半旬から<br>{em}月 第{ed}半旬 の値を<br>{sy}年から {ey}年まで表示"
    elif pc == "4":  # 旬別値
        sdn, edn = _decad_name(int(sd)), _decad_name(int(ed))
        if mode_dd.value == "1":
            text = f"{sy}年{sm}月{sdn}から<br>{ey}年{em}月{edn}<br>まで の旬別値を表示"
        else:
            text = f"{sm}月{sdn}から<br>{em}月{edn}の値を<br>{sy}年から{ey}年まで表示"
    elif pc == "5":  # 月別値
        if mode_dd.value == "1":
            text = f"{sy}年{sm}月から<br>{ey}年{em}月まで の月別値を表示"
        else:
            text = f"{sm}月から{em}月の値を<br>{sy}年から{ey}年まで表示"
    elif pc == "6":  # 3か月別値
        sq, eq = _quarter_label(sm), _quarter_label(em)
        if mode_dd.value == "1":
            text = f"{sy}年{sq}から<br>{ey}年{eq}まで の3か月別値を表示"
        else:
            text = f"{sq}から{eq}<br>{sy}年から{ey}年まで表示"
    else:  # 時別値・日別値
        if mode_dd.value == "1":
            text = f"{sy}年{sm}月{sd}日から<br>{ey}年{em}月{ed}日まで の{period_label}を表示"
        else:
            text = f"{sm}月{sd}日から{em}月{ed}日の値を<br>{sy}年から{ey}年まで表示"

    selected_period_label.value = (
        "<div style='margin-top:4px;padding:6px 8px;background:#F4F9F9;"
        "border:1px solid #D6E8E8;font-size:12.5px;color:#333;line-height:1.5'>"
        "<b>選択された期間（日本標準時）</b><br>" + text + "</div>")


def refresh_period(*_):
    refresh_period_fields()
    refresh_period_summary()


period_dd.observe(refresh_elements, names="value")
period_dd.observe(refresh_period, names="value")
mode_dd.observe(refresh_desc, names="value")
mode_dd.observe(refresh_period_summary, names="value")
for _dd in (syear, smonth, sday, eyear, emonth, eday):
    _dd.observe(refresh_period_summary, names="value")
refresh_elements()
refresh_period_fields()
refresh_period_summary()
refresh_desc()


def set_status(html):
    status_label.value = f"<span style='font-size:13px'>{html}</span>"


def run_10min_download(stations: List[Station], inter_type: str, y1, m1, d1, y2, m2, d2):
    """10分値の取得本体。obsdl系（build_plan・fetch_data等）とは完全に別経路（etrnスクレイピング）。"""
    if inter_type == "2":
        target_dates = list(dates_multi_year(y1, y2, m1, d1, m2, d2))
    else:
        target_dates = list(dates_in_range(y1, m1, d1, y2, m2, d2))
    if not target_dates:
        print("有効な期間がありません。")
        return

    n_requests = len(stations) * len(target_dates)
    est_min = n_requests * SLEEP_SEC / 60.0

    print("── 設定内容 ──")
    print("  データの種類 : 10分値（気象庁「過去の気象データ検索」から取得）")
    print(f"  対象地点数   : {len(stations)} 地点")
    print(f"  対象日数     : {len(target_dates)} 日")
    print(f"  リクエスト数 : {n_requests} 回（地点×日、間隔 {SLEEP_SEC}秒 → 推定 約 {est_min:.1f} 分）")
    print("  まとめ方     : 地点ごとに1ファイル（10分値は常にこの形式）")
    if n_requests > 2000:
        print("  ⚠ 件数が多いため取得にかなり時間がかかります。期間や地点を絞ることをおすすめします。")
    print()

    raw_dir = Path("jma_raw")
    merged_dir = Path("jma_data")
    for _d in (raw_dir, merged_dir):
        if _d.exists():
            shutil.rmtree(_d)
        _d.mkdir()

    progress.max = max(1, n_requests)
    progress.layout.visibility = "visible"
    set_status("🌐 気象庁へ接続中…")
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"})
    session.get(ROOT_URL, timeout=30)
    time.sleep(SLEEP_SEC)

    n_ok = n_fail = n_empty = 0
    done = 0
    failures: List[str] = []
    start_time = time.time()
    station_lines: Dict[str, List[str]] = {}
    station_header: Dict[str, str] = {}

    def update_status(current_name):
        elapsed = time.time() - start_time
        eta = ""
        if done > 0:
            remain = (elapsed / done) * (n_requests - done)
            eta = f"・残り 約 {remain/60:.1f} 分"
        set_status(
            f"📥 取得中: <b>{current_name}</b>　{done}/{n_requests} 件"
            f"（成功 {n_ok}／欠測 {n_empty}／失敗 {n_fail}）{eta}"
        )

    for st in stations:
        for dt in target_dates:
            disp = f"{st.name} {dt.isoformat()}"
            print(f"取得中: {disp}")
            update_status(disp)
            try:
                result = fetch_10min_day(session, st, dt.year, dt.month, dt.day)
                if result is None:
                    print("  データなし（欠測、または未観測日）")
                    n_empty += 1
                else:
                    labels, data_rows = result
                    header_line = ",".join(_csv_escape(c) for c in (["年月日時分"] + labels[1:]))
                    lines = station_lines.setdefault(st.station_id, [])
                    station_header.setdefault(st.station_id, header_line)
                    date_str = f"{dt.year}/{dt.month}/{dt.day}"
                    for row in data_rows:
                        lines.append(",".join(_csv_escape(c) for c in ([f"{date_str} {row[0]}"] + row[1:])))
                    n_ok += 1
            except Exception as e:
                print(f"  [エラー] {disp}: {e}")
                failures.append(f"{disp}: {e}")
                n_fail += 1
            done += 1
            progress.value = done
            update_status(disp)
            time.sleep(SLEEP_SEC)

    set_status("🗂 地点ごとにまとめています…")
    for st in stations:
        lines = station_lines.get(st.station_id)
        if not lines:
            continue
        try:
            write_lines([station_header[st.station_id]] + lines, merged_dir / f"{st.name}_10分値.csv")
        except Exception as e:
            print(f"  [結合エラー] {st.name}: {e}")

    progress.bar_style = "success" if n_fail == 0 else "warning"
    set_status(f"✅ 完了：成功 {n_ok}／欠測 {n_empty}／失敗 {n_fail}　（下の枠に詳細）")
    print()
    print("── 結果サマリ ──")
    print(f"  成功: {n_ok} / 欠測: {n_empty} / 失敗: {n_fail}")
    if failures:
        print("  失敗した項目:")
        for f in failures[:50]:
            print(f"    - {f}")
        if len(failures) > 50:
            print(f"    …ほか {len(failures) - 50} 件")

    has_files = any(merged_dir.iterdir())
    if has_files:
        print()
        print("ZIPにまとめてダウンロードします...")
        shutil.make_archive("jma_data_result", "zip", merged_dir)
        files.download("jma_data_result.zip")
    else:
        print("保存できたファイルがないため、ZIPは作成しませんでした。")


def on_run(_btn):
    run_btn.disabled = True
    progress.value = 0
    progress.bar_style = "info"
    progress.layout.visibility = "hidden"
    set_status("⏳ 準備中…（設定を確認しています）")
    out.clear_output()
    completed = False
    with out:
        try:
            csv_text = paste_area.value.strip()
            if not csv_text:
                print("地点リストを貼り付け欄に貼り付けてください（ファインダーの「Colab用にコピー」）。")
                return
            stations = read_stations_from_text(csv_text)
            if not stations:
                print("有効な地点がCSVから読み取れませんでした。")
                return

            inter_type = mode_dd.value
            period_code = period_dd.value
            period_label = dict((c, l) for c, l, _u in PERIOD_OPTIONS)[period_code]
            sleep_sec = SLEEP_SEC

            y1, m1, d1 = int(syear.value), int(smonth.value), int(sday.value)
            y2, m2, d2 = int(eyear.value), int(emonth.value), int(eday.value)

            # 期間の妥当性チェック
            if inter_type == "2":
                if y1 > y2:
                    print("開始年は終了年以前にしてください。")
                    return
            else:
                if date(y1, m1, min(d1, calendar.monthrange(y1, m1)[1])) > date(y2, m2, min(d2, calendar.monthrange(y2, m2)[1])):
                    print("開始日は終了日以前にしてください。")
                    return

            if period_code == "10min":
                run_10min_download(stations, inter_type, y1, m1, d1, y2, m2, d2)
                completed = True
                return

            chunk_unit = dict((c, u) for c, _l, u in PERIOD_OPTIONS)[period_code]
            element_codes = [code for code, cb in elem_checks if cb.value]
            if not element_codes:
                print("観測項目を1つ以上選択してください。")
                return
            sel_cats = {CODE_TO_CATEGORY.get((period_code, c), "") for c in element_codes}
            sel_cats.discard("")

            aggrg_type = int(period_code)

            # リクエスト計画（期間軸: 気象庁の1回あたりデータ量上限に収まるよう期間を分割）
            plan = build_plan(inter_type, aggrg_type, len(element_codes), y1, m1, d1, y2, m2, d2, chunk_unit)
            n_el = len(element_codes)

            # 出力フォルダは毎回まっさらにする（前回の実行分がZIPに混ざらないように）
            raw_dir = Path("jma_raw")
            merged_dir = Path("jma_data")
            for _d in (raw_dir, merged_dir):
                if _d.exists():
                    shutil.rmtree(_d)
                _d.mkdir()

            # 1回のリクエストに何地点まとめられるか。地点数×項目数×点数 ≤ 上限。
            def stations_per_request(inter, ymd):
                cost = max(1, n_el * count_periods(aggrg_type, int(inter), *ymd))
                return max(1, VOLUME_LIMIT // cost)

            # 気象庁への問い合わせは、まとめ方の選択に関わらず常に「地点をなるべくまとめて
            # リクエスト数を最小化する」方式で行う。出力（地点ごと／期間ごと／1ファイル）は
            # 取得後にローカルでCSVを分割・結合して作る（fetch_planが取得計画、
            # station_posは各期間区分内での各地点の位置＝どのバッチの何番目か）。
            fetch_plan = []
            for inter, ymd, suffix in plan:
                spr = stations_per_request(inter, ymd)
                batches = [stations[i:i + spr] for i in range(0, len(stations), spr)]
                station_pos = {}
                for bi, batch in enumerate(batches):
                    for k, st in enumerate(batch):
                        station_pos[st.station_id] = (bi, k, len(batch))
                fetch_plan.append({"suffix": suffix, "inter": inter, "ymd": ymd,
                                    "batches": batches, "station_pos": station_pos})

            mode = merge_mode.value
            if mode == "auto":
                # ファイルが少ない方を選ぶ（地点ごと＝地点数、期間ごと＝期間区分数）。
                mode = "period" if len(fetch_plan) <= len(stations) else "station"

            reqs = []
            raw_paths: Dict[Tuple[int, int], Path] = {}
            for fi, fp in enumerate(fetch_plan):
                n_batches = len(fp["batches"])
                for bi, batch in enumerate(fp["batches"]):
                    out_path = (raw_dir / f"{period_label}_{fp['suffix']}_p{bi + 1}.csv" if n_batches > 1
                                else raw_dir / f"{period_label}_{fp['suffix']}.csv")
                    raw_paths[(fi, bi)] = out_path
                    disp = (f"{fp['suffix']}（{len(batch)}地点／グループ{bi + 1}/{n_batches}）" if n_batches > 1
                            else f"{fp['suffix']}（{len(batch)}地点）")
                    reqs.append({"nums": [b.station_num() for b in batch], "inter": fp["inter"], "ymd": fp["ymd"],
                                 "path": out_path, "display": disp, "fi": fi, "bi": bi})

            n_requests = len(reqs)
            est_min = n_requests * sleep_sec / 60.0
            discontinued = [s for s in stations if s.status and s.status != "現役"]
            mismatch = [s.name for s in stations if s.elements and sel_cats and not (sel_cats & s.elements)]

            print("── 設定内容 ──")
            print(f"  期間の種類   : {'特定の期間を複数年分' if inter_type == '2' else '連続した期間'}")
            print(f"  データの種類 : {period_label}")
            print(f"  観測項目     : {', '.join(element_codes)}")
            if inter_type == "2":
                print(f"  期間         : 各年 {m1}/{d1} 〜 {m2}/{d2} を {y1}年〜{y2}年")
            else:
                print(f"  期間         : {y1}/{m1}/{d1} 〜 {y2}/{m2}/{d2}（{'1か月' if chunk_unit=='month' else '1年'}ごと）")
            print(f"  対象地点数   : {len(stations)} 地点")
            mode_label = {"station": "地点ごと", "period": "期間ごと", "single": "常に1ファイルにまとめる"}[mode]
            print(f"  まとめ方     : {mode_label}"
                  f"{'（自動選択）' if merge_mode.value == 'auto' else ''}"
                  "（取得後にローカルで適用。問い合わせ回数には影響しません）")
            print(f"  リクエスト数 : 約 {n_requests} 回（間隔 {sleep_sec}秒 → 推定 約 {est_min:.1f} 分）")
            if len(plan) > 1:
                print(f"    ※ 気象庁の1回あたりデータ量上限を超えないよう自動分割しています")
            if discontinued:
                print(f"  ⚠ 廃止済み地点が {len(discontinued)} 件（期間により空データの場合あり）")
            if mismatch:
                ex = "、".join(mismatch[:5]) + ("…" if len(mismatch) > 5 else "")
                print(f"  ⚠ 選んだ項目を観測していない可能性のある地点 {len(mismatch)} 件（例: {ex}）")
            print()

            progress.max = max(1, n_requests)
            progress.layout.visibility = "visible"
            set_status("🌐 気象庁へ接続中…")
            session = requests.Session()
            session.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"})
            session.get(ROOT_URL, timeout=30)
            time.sleep(sleep_sec)

            n_ok = n_fail = 0
            done = 0
            failures: List[str] = []
            start_time = time.time()

            def update_status(current_name):
                elapsed = time.time() - start_time
                eta = ""
                if done > 0:
                    remain = (elapsed / done) * (n_requests - done)
                    eta = f"・残り 約 {remain/60:.1f} 分"
                set_status(
                    f"📥 取得中: <b>{current_name}</b>　{done}/{n_requests} 件"
                    f"（成功 {n_ok}／失敗 {n_fail}）{eta}"
                )

            ok_batches: Dict[Tuple[int, int], bool] = {}
            for req in reqs:
                out_path = req["path"]
                fname = out_path.name
                print(f"取得中: {req['display']}")
                update_status(req["display"])
                try:
                    content = fetch_data(session, req["nums"], req["ymd"], period_code,
                                         element_codes, req["inter"], sleep_sec)
                    save_csv(content, out_path)
                    ok_batches[(req["fi"], req["bi"])] = True
                    print(f"  保存: {fname}")
                    n_ok += 1
                except Exception as e:
                    print(f"  [エラー] {fname}: {e}")
                    failures.append(f"{fname}: {e}")
                    n_fail += 1
                done += 1
                progress.value = done
                update_status(req["display"])
                time.sleep(sleep_sec)

            # 期間区分（fi）ごとに、取得できた地点グループ（列）をすべて結合し、
            # その期間区分における「全地点ぶんの行リスト」を作る（未保存・メモリ上）。
            def period_lines(fi):
                fp = fetch_plan[fi]
                paths = [raw_paths[(fi, bi)] for bi in range(len(fp["batches"])) if ok_batches.get((fi, bi))]
                if not paths:
                    return None
                return merge_columns([_read_lines(p) for p in paths])

            if mode == "period":
                set_status("🗂 期間ごとにまとめています…")
                for fi, fp in enumerate(fetch_plan):
                    lines = period_lines(fi)
                    if lines is None:
                        continue
                    try:
                        write_lines(lines, merged_dir / f"{period_label}_{fp['suffix']}.csv")
                    except Exception as e:
                        print(f"  [結合エラー] {fp['suffix']}: {e}")

            elif mode == "single":
                set_status("🗂 1ファイルにまとめています…")
                chunks = [c for c in (period_lines(fi) for fi in range(len(fetch_plan))) if c is not None]
                if chunks:
                    try:
                        write_lines(merge_rows(chunks), merged_dir / f"{period_label}.csv")
                    except Exception as e:
                        print(f"  [結合エラー] 1ファイルまとめ: {e}")

            else:  # station: 地点ごとに、各期間区分の該当列だけを抜き出して行方向に結合する
                set_status("🗂 地点ごとにまとめています…")
                for s in stations:
                    chunks = []
                    for fi, fp in enumerate(fetch_plan):
                        pos = fp["station_pos"].get(s.station_id)
                        if pos is None:
                            continue
                        bi, idx, n_in_batch = pos
                        if not ok_batches.get((fi, bi)):
                            continue
                        lines = _read_lines(raw_paths[(fi, bi)])
                        chunks.append(_station_slice(lines, idx, n_in_batch))
                    if not chunks:
                        continue
                    try:
                        write_lines(merge_rows(chunks), merged_dir / f"{s.name}_{period_label}.csv")
                    except Exception as e:
                        print(f"  [結合エラー] {s.name}: {e}")

            progress.bar_style = "success" if n_fail == 0 else "warning"
            set_status(f"✅ 完了：成功 {n_ok}／失敗 {n_fail}　（下の枠に詳細）")
            print()
            print("── 結果サマリ ──")
            print(f"  成功: {n_ok} / 失敗: {n_fail}")
            if failures:
                print("  失敗した項目:")
                for f in failures:
                    print(f"    - {f}")

            has_files = any(merged_dir.iterdir())
            if has_files:
                print()
                print("ZIPにまとめてダウンロードします...")
                shutil.make_archive("jma_data_result", "zip", merged_dir)
                files.download("jma_data_result.zip")
            else:
                print("保存できたファイルがないため、ZIPは作成しませんでした。")
            completed = True
        finally:
            run_btn.disabled = False
            if not completed:
                progress.layout.visibility = "hidden"
                set_status("⚠️ 中断しました。下の枠のメッセージをご確認ください。")


run_btn.on_click(on_run)

def _section(title):
    return widgets.HTML(f"<div style='margin:10px 0 2px;font-weight:700;color:#1C7C8C;"
                        f"border-bottom:2px solid #E4F0F1;padding-bottom:2px'>{title}</div>")

form = widgets.VBox([
    widgets.HTML("<h3 style='margin:4px 0'>気象庁データ 一括ダウンロード</h3>"
                 "<div style='color:#555;font-size:13px'>下の 1 → 2 → 3 の順に選び、「ダウンロード開始」を押してください。</div>"),

    _section("1. 地点リストを貼り付け"),
    paste_area,

    _section("2. データの種類 ＆ 観測項目"),
    period_dd,
    elem_label,
    elem_box,
    elem_note_10min,

    _section("3. 期間の種類 ＆ 開始〜終了"),
    mode_dd,
    period_desc,
    start_row,
    end_row,
    selected_period_label,

    _section("まとめ方"),
    merge_mode,
    merge_hint,
    merge_note_10min,

    run_btn,
    progress,
    status_label,
    out,
], layout=widgets.Layout(max_width="640px"))

display(form)
